# GroupBy and Aggregation

**Learning Objectives:**
- Use `groupby()` to split data into groups
- Aggregate with `agg()` using dicts, lists, and custom functions
- Transform and filter grouped data
- Understand `apply()` vs `agg()`
- Create pivot tables with `pivot_table()` and `crosstab()`
- Work with multi-level grouped DataFrames

## 1. Setup – Sales Dataset

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)
sales = pd.DataFrame({
    "Region": np.random.choice(["North", "South", "East", "West"], 50),
    "Product": np.random.choice(["A", "B", "C"], 50),
    "Sales": np.random.randint(100, 1000, 50),
    "Quantity": np.random.randint(1, 20, 50)
})
print(sales.head(10))
print("\nShape:", sales.shape)

## 2. Basic `groupby()` + Aggregation

Split data into groups and compute summary statistics.

In [ ]:
# Group by Region, get mean Sales
region_mean = sales.groupby("Region")["Sales"].mean()
print("Mean Sales by Region:")
print(region_mean)

# Group by multiple columns
multiple = sales.groupby(["Region", "Product"])["Sales"].sum()
print("\nSum Sales by Region and Product:")
print(multiple)

In [ ]:
# Multiple aggregation functions
agg_result = sales.groupby("Region")["Sales"].agg(["sum", "mean", "std", "count"])
print("Multiple aggregations:")
print(agg_result)

## 3. `agg()` with Dict – Different Functions per Column

In [ ]:
# Different functions for different columns
agg_dict = sales.groupby("Region").agg({
    "Sales": ["sum", "mean"],
    "Quantity": ["sum", "count"]
})
print(agg_dict)

## 4. Custom Aggregation Functions

In [ ]:
# Custom function: range (max - min)
def value_range(x):
    return x.max() - x.min()

custom = sales.groupby("Region")["Sales"].agg(["mean", value_range])
print(custom)

## 5. `transform()` – Same Shape as Original

Returns a DataFrame/Series with the same shape, useful for creating new columns.

In [ ]:
# Add each row's region mean as a new column
sales["Region_Mean_Sales"] = sales.groupby("Region")["Sales"].transform("mean")
print(sales.head(10))

## 6. `filter()` – Keep Groups Based on Condition

Removes entire groups that don't satisfy a condition.

In [ ]:
# Keep only regions with total Sales > 2000
filtered = sales.groupby("Region").filter(lambda g: g["Sales"].sum() > 2000)
print("Regions with total Sales > 2000:")
print(filtered["Region"].unique())
print("\nShape after filter:", filtered.shape)

## 7. `apply()` vs `agg()`

- `agg()` – returns a **scalar** per group per function
- `apply()` – can return a scalar, Series, or DataFrame (most flexible)

In [ ]:
# apply: return a DataFrame for each group
top2 = sales.groupby("Region").apply(lambda g: g.nlargest(2, "Sales"))
print("Top 2 sales per region:")
print(top2)

## 8. `pivot_table()` – Spreadsheet-Style Pivot

Reshape data: index, columns, values, aggfunc.

In [ ]:
# Pivot: rows=Region, cols=Product, values=Sales, aggfunc=sum
pivot = pd.pivot_table(sales, index="Region", columns="Product", values="Sales", aggfunc="sum")
print("Pivot Table:")
print(pivot)

# With margins (row/column totals)
pivot_margins = pd.pivot_table(sales, index="Region", columns="Product",
                                values="Sales", aggfunc="sum", margins=True)
print("\nWith margins:")
print(pivot_margins)

## 9. `crosstab()` – Frequency Tables

Cross-tabulation: counts or proportions of two (or more) factors.

In [ ]:
# Frequency of Region vs Product
ct = pd.crosstab(sales["Region"], sales["Product"])
print("Cross-tabulation (counts):")
print(ct)

# With normalize for proportions
ct_pct = pd.crosstab(sales["Region"], sales["Product"], normalize="index")
print("\nRow proportions:")
print(ct_pct)

## 10. Practice Problems

In [ ]:
# Problem 1: For each Product, compute total Sales, average Quantity,
#            and number of orders. Use agg() with a dict.
product_stats = sales.groupby("Product").agg({
    "Sales": "sum",
    "Quantity": ["mean", "count"]
})
print("Product statistics:")
print(product_stats)

In [ ]:
# Problem 2: Create a pivot table showing average Quantity for each
#            Region (rows) and Product (columns).
pivot_qty = pd.pivot_table(sales, index="Region", columns="Product",
                            values="Quantity", aggfunc="mean")
print(pivot_qty)

In [ ]:
# Problem 3: Use transform() to add a column 'Pct_of_Region_Total'
#            showing what percentage each sale contributes to its region.
region_total = sales.groupby("Region")["Sales"].transform("sum")
sales["Pct_of_Region_Total"] = (sales["Sales"] / region_total) * 100
print(sales.head(10))

## 11. Summary

| Method | Output Shape | Use Case |
|--------|-------------|----------|
| `groupby()` + aggregation | Reduced | Summary per group |
| `agg()` | Reduced | Multiple functions per column |
| `transform()` | Same as input | Add group-derived columns |
| `filter()` | Subset of rows | Remove entire groups |
| `apply()` | Flexible | Complex group-wise logic |
| `pivot_table()` | Reshaped | Spreadsheet-like summaries |
| `crosstab()` | Reshaped | Frequency / proportion tables |